# Laborator 2

În Colab: *Runtime → Change runtime type → T4 GPU* (opțional, rețelele sunt mici și merg și pe CPU).

## Exerciții

### 1. Dataset MNIST vectorizat
Fie setul de date MNIST ce conține imagini cu cifre de la 0 la 9.

Completați metoda `__getitem__(self, idx)` a Dataset-ului astfel încât să returneze imaginea vectorizată și scalată în [0, 1] precum și label-ul asociat.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)

train_data = datasets.MNIST(root="./data", train=True, download=True)
test_data = datasets.MNIST(root="./data", train=False, download=True)

X_train = train_data.data.numpy()
y_train = train_data.targets.numpy()

X_test = test_data.data.numpy()
y_test = test_data.targets.numpy()

print(X_train.shape, X_train.dtype, y_train.shape)
print(X_test.shape, X_test.dtype, y_test.shape)

100%|██████████| 9.91M/9.91M [00:10<00:00, 966kB/s] 
100%|██████████| 28.9k/28.9k [00:00<00:00, 226kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.13MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.54MB/s]

(60000, 28, 28) uint8 (60000,)
(10000, 28, 28) uint8 (10000,)


In [2]:
class MNISTVector(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # (28, 28) uint8 -> (784,) float32 in [0, 1]; label -> int64
        x = self.images[idx].flatten().astype(np.float32) / 255.0
        y = self.labels[idx].astype(np.int64)
        return x, y


train_dataset = MNISTVector(X_train, y_train)
test_dataset = MNISTVector(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

for X, y in train_loader:
    print(X.shape, y.shape)
    break

assert X.shape == (50, 784) and y.shape == (50,)
assert X.dtype == torch.float32 and y.dtype == torch.int64
assert X.min() >= 0 and X.max() <= 1

torch.Size([50, 784]) torch.Size([50])


**Hint**:

`torch.from_numpy(...)` / `tensor.float()`

`tensor.view(-1)` / `tensor.flatten()`

Pixelii sunt de tip `uint8`, cu valori în [0, 255].

`nn.CrossEntropyLoss` așteaptă label-uri de tip `torch.long`.

### 2. Rețea complet conectată (MLP)
Implementați o rețea simplă pentru a procesa imaginile vectorizate.
- Rețeaua trebuie să conțină doar straturi complet conectate (`nn.Linear(...)`).
- Rețeaua trebuie să conțină cel puțin 2 straturi (inclusiv ultimul).
- După ultimul strat nu urmează activare.
- După orice alt strat puteți să folosiți fie activare sigmoid (`F.sigmoid(...)`) sau tanh (`F.tanh(...)`) sau relu (`F.relu(...)`).
- Parametrii rețelei se scriu în constructor (`self.fc1 = nn.Linear(...)`).
- Parcurgerea rețelei se scrie în metoda `forward`.
- Intrarea rețelei este un tensor de dimensiunea (B, 784) unde B este batch_size și 784 reprezintă dimensiunea unui vector anterior calculat.
- Ieșirea rețelei este un tensor de dimensiune (B, 10) unde B este batch_size și 10 reprezintă numărul de clase.

In [3]:
class MLP(nn.Module):
    def __init__(self, in_features=784, hidden=128, n_classes=10, act=F.relu):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, n_classes)
        self.act = act

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP().to(device)
print(device)
print(model)

for X, y in train_loader:
    X, y = X.to(device), y.to(device)
    y_pred = model(X)
    print(y_pred.shape)
    break

assert y_pred.shape == (X.shape[0], 10)
assert sum(isinstance(m, nn.Linear) for m in model.modules()) >= 2

cpu
MLP(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)
torch.Size([50, 10])


**Hint**:

`self.fc1 = nn.Linear(in_features, hidden)`, `self.fc2 = nn.Linear(hidden, n_classes)`

Păstrați activarea primită ca parametru (`self.act = act`) și aplicați-o în `forward` după fiecare strat, mai puțin ultimul.

Parametrii `in_features`, `hidden` și `act` sunt folosiți la ex. 3 și 6.

### 3. Antrenare
Antrenați modelul și observați rezultatele. Influențează funcțiile de activare rezultatele? Dar numărul de neuroni?

Funcția `test` se scrie la ex. 4, așa că aici bucla apelează doar `train`.

In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def train(model, loader, optimizer):
    model.train()

    total_loss = 0
    total_correct = 0
    total_data = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()

        out = model(X)
        loss = criterion(out, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        total_correct += (out.argmax(dim=-1) == y).sum().item()
        total_data += y.size(0)

    return total_loss / len(loader), total_correct / total_data


epochs = 10
for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer)
    print(f"Epoch {epoch}:")
    print(f"train loss={train_loss:.4f}, train acc={train_acc:.4f}")

Epoch 0:
train loss=2.1828, train acc=0.3521
Epoch 1:
train loss=1.8694, train acc=0.6577
Epoch 2:
train loss=1.4808, train acc=0.7400
Epoch 3:
train loss=1.1517, train acc=0.7936
Epoch 4:
train loss=0.9300, train acc=0.8221
Epoch 5:
train loss=0.7873, train acc=0.8376
Epoch 6:
train loss=0.6922, train acc=0.8495
Epoch 7:
train loss=0.6257, train acc=0.8573
Epoch 8:
train loss=0.5770, train acc=0.8633
Epoch 9:
train loss=0.5398, train acc=0.8690


Funcție ajutătoare, folosită și în exercițiile următoare: antrenează un număr fix de epoci și, dacă primește `test_loader`, afișează și metricile pe testare (cu `test` de la ex. 4).

In [5]:
def fit(model, train_loader, test_loader=None, epochs=10, optimizer=None, scheduler=None, verbose=True):
    model.to(device)
    if optimizer is None:
        optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

    for epoch in range(epochs):
        train_loss, train_acc = train(model, train_loader, optimizer)
        if scheduler is not None:
            scheduler.step()

        msg = f"Epoch {epoch}: train loss={train_loss:.4f}, acc={train_acc:.4f}"
        if test_loader is not None:  # reporting only, never used for training
            test_loss, test_acc = test(model, test_loader)
            msg += f" | test loss={test_loss:.4f}, acc={test_acc:.4f}"
        if verbose:
            print(msg)

    return train_loss, train_acc

In [13]:
# Same optimizer / lr / epochs / seed for every run; change one factor at a time
exp_epochs = 5

for name, act in [{"sigmoid": F.sigmoid,
                   "tanh": F.tanh,
                   "relu": F.relu}]:  # sigmoid / tanh / relu
    torch.manual_seed(0)
    loss, acc = fit(MLP(act=act), train_loader, test_loader, epochs=exp_epochs, optimizer=torch.optim.SGD(MLP(act=act).parameters(), lr=1e-3))
    print(f"act={name:<7} hidden=128 : train loss={loss:.4f}, acc={acc:.4f}")

for hidden in [16, 64, 265, 1024]:  # e.g. 16, 64, 256, 1024
    torch.manual_seed(0)
    loss, acc = fit(MLP(hidden=hidden), train_loader, test_loader, epochs=exp_epochs, optimizer=torch.optim.SGD(MLP(hidden=hidden).parameters(), lr=1e-3))
    print(f"act=relu    hidden={hidden:<4}: train loss={loss:.4f}, acc={acc:.4f}")

ValueError: too many values to unpack (expected 2)

**Hint**:

`fit(MLP(hidden=..., act=...), train_loader, epochs=exp_epochs, verbose=False)` întoarce `(train_loss, train_acc)` de la ultima epocă.

**Observații**:

- Funcția de activare: ...
- Numărul de neuroni: ...

### 4. Evaluare pe datele de testare
Scrieți o funcție similară pentru a calcula funcția de cost și acuratețea pe datele de testare și apelați-o după pasul de antrenare.

In [33]:
def test(model, loader):
    model.eval()

    total_loss = 0
    total_correct = 0
    total_data = 0

    with torch.no_grad():
        for X, y in loader:
            X.to(device), y.to(device)

            out = model(X)
            loss = criterion(out, y)
            
            total_loss += loss.item()
            
            total_correct += (out.argmax(dim=-1) == y).sum().item()
            total_data += y.size(0)


    return total_loss / len(loader), total_correct / total_data

In [15]:
torch.manual_seed(0)
model = MLP().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, optimizer)
    test_loss, test_acc = test(model, test_loader)
    print(f"Epoch {epoch}:")
    print(f"train loss={train_loss:.4f}, train acc={train_acc:.4f}")
    print(f"test  loss={test_loss:.4f}, test  acc={test_acc:.4f}")

assert 0.5 < test_acc <= 1

Epoch 0:
train loss=2.1720, train acc=0.4303
test  loss=0.0102, test  acc=0.6400
Epoch 1:
train loss=1.8393, train acc=0.6882
test  loss=0.0083, test  acc=0.7200
Epoch 2:
train loss=1.4498, train acc=0.7529
test  loss=0.0063, test  acc=0.7400
Epoch 3:
train loss=1.1275, train acc=0.7913
test  loss=0.0049, test  acc=0.7800
Epoch 4:
train loss=0.9131, train acc=0.8198
test  loss=0.0040, test  acc=0.8200
Epoch 5:
train loss=0.7761, train acc=0.8373
test  loss=0.0034, test  acc=0.8200
Epoch 6:
train loss=0.6849, train acc=0.8481
test  loss=0.0030, test  acc=0.8400
Epoch 7:
train loss=0.6206, train acc=0.8563
test  loss=0.0027, test  acc=0.8400
Epoch 8:
train loss=0.5733, train acc=0.8632
test  loss=0.0025, test  acc=0.8400
Epoch 9:
train loss=0.5370, train acc=0.8689
test  loss=0.0024, test  acc=0.8600


**Hint**:

Aceeași buclă ca în `train`, dar fără `optimizer.zero_grad()`, `loss.backward()` și `optimizer.step()`.

`model.eval()` + `torch.no_grad()`: nu se calculează gradienți, iar Dropout / BatchNorm trec în modul de evaluare (contează la ex. 9).

### 5. Normalizare pe trăsături (medie 0, deviație 1)
Pentru datele de antrenare, înainte de a antrena rețeaua, calculați media și deviația pentru fiecare din cele 784 trăsături, după ce au fost scalate. Folosiți vectorii de medie și deviație pentru a normaliza datele în clasa `MNISTVector`. Țineți cont de faptul că datele de testare trebuie să parcurgă aceleași etape de preprocesare ca datele de antrenare.

In [24]:
X_train_scaled = X_train.reshape(len(X_train), -1).astype(np.float32) / 255.0
mean = X_train_scaled.mean(axis=0)
std = X_train_scaled.std(axis=0)
std[std == 0] = 1.0

In [25]:
class MNISTVectorNORM(Dataset):
    def __init__(self, images, labels, mean=mean, std=std):
        self.images = images
        self.labels = labels
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # (28, 28) uint8 -> (784,) float32 in [0, 1]; label -> int64
        x = self.images[idx].flatten().astype(np.float32) / 255.0
        x = (x - self.mean) / self.std
        y = self.labels[idx].astype(np.int64)
        return x, y


train_dataset1 = MNISTVectorNORM(X_train, y_train)
test_dataset1 = MNISTVectorNORM(X_test, y_test)

train_loader1 = DataLoader(train_dataset1, batch_size=50, shuffle=True)
test_loader1 = DataLoader(test_dataset1, batch_size=50, shuffle=False)

for X, y in train_loader1:
    print(X.shape, y.shape)
    break

torch.Size([50, 784]) torch.Size([50])


In [26]:
torch.manual_seed(0)
model = MLP().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader1, optimizer)
    test_loss, test_acc = test(model, test_loader1)
    print(f"Epoch {epoch}:")
    print(f"train loss={train_loss:.4f}, train acc={train_acc:.4f}")
    print(f"test  loss={test_loss:.4f}, test  acc={test_acc:.4f}")

Epoch 0:
train loss=1.7911, train acc=0.5519
test  loss=0.0064, test  acc=0.6800
Epoch 1:
train loss=1.0615, train acc=0.7640
test  loss=0.0041, test  acc=0.8400
Epoch 2:
train loss=0.7131, train acc=0.8339
test  loss=0.0029, test  acc=0.8800
Epoch 3:
train loss=0.5495, train acc=0.8662
test  loss=0.0023, test  acc=0.9000
Epoch 4:
train loss=0.4629, train acc=0.8822
test  loss=0.0020, test  acc=0.9000
Epoch 5:
train loss=0.4102, train acc=0.8923
test  loss=0.0017, test  acc=0.9200
Epoch 6:
train loss=0.3747, train acc=0.8991
test  loss=0.0016, test  acc=0.9200
Epoch 7:
train loss=0.3488, train acc=0.9041
test  loss=0.0015, test  acc=0.9200
Epoch 8:
train loss=0.3289, train acc=0.9086
test  loss=0.0014, test  acc=0.9200
Epoch 9:
train loss=0.3130, train acc=0.9122
test  loss=0.0013, test  acc=0.9200


### 6. PCA (100 de trăsături)
După normalizarea de la 5., pentru datele de antrenare aplicați PCA pentru a obține 100 de trăsături. Țineți cont de faptul că datele de testare trebuie să parcurgă aceleași etape de preprocesare ca datele de antrenare.

In [28]:
from sklearn.decomposition import PCA

# Date normalizate folosind mean/std calculate pe train
X_train_norm = (X_train.reshape(len(X_train), -1).astype(np.float32) / 255.0 - mean) / std
X_test_norm = (X_test.reshape(len(X_test), -1).astype(np.float32) / 255.0 - mean) / std

# PCA se antrenează exclusiv pe datele de train
pca = PCA(n_components=100, random_state=0)

X_train_pca = pca.fit_transform(X_train_norm).astype(np.float32)
X_test_pca = pca.transform(X_test_norm).astype(np.float32)

print(X_train_pca.shape)  # (60000, 100)
print(X_test_pca.shape)   # (10000, 100)

(60000, 100)
(10000, 100)


In [29]:
class PCA_Dataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.from_numpy(images)
        self.labels = torch.from_numpy(labels.astype(np.int64))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]


train_dataset_pca = PCA_Dataset(X_train_pca, y_train)
test_dataset_pca = PCA_Dataset(X_test_pca, y_test)

train_loader_pca = DataLoader(train_dataset_pca, batch_size=50, shuffle=True)
test_loader_pca = DataLoader(test_dataset_pca, batch_size=50, shuffle=False)

In [31]:
class MLP_PCA(nn.Module):
    def __init__(self, in_features=100, hidden=64, n_classes=10, act=F.relu):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, n_classes)
        self.act = act

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP_PCA().to(device)
print(device)
print(model)

cpu
MLP_PCA(
  (fc1): Linear(in_features=100, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=10, bias=True)
)


In [34]:
torch.manual_seed(0)
model = MLP_PCA().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader_pca, optimizer)
    test_loss, test_acc = test(model, test_loader_pca)
    print(f"Epoch {epoch}:")
    print(f"train loss={train_loss:.4f}, train acc={train_acc:.4f}")
    print(f"test  loss={test_loss:.4f}, test  acc={test_acc:.4f}")

Epoch 0:
train loss=1.7739, train acc=0.4899
test  loss=1.2771, test  acc=0.7321
Epoch 1:
train loss=1.0152, train acc=0.7730
test  loss=0.7888, test  acc=0.8242
Epoch 2:
train loss=0.6972, train acc=0.8343
test  loss=0.5893, test  acc=0.8613
Epoch 3:
train loss=0.5544, train acc=0.8625
test  loss=0.4907, test  acc=0.8767
Epoch 4:
train loss=0.4769, train acc=0.8763
test  loss=0.4333, test  acc=0.8857
Epoch 5:
train loss=0.4287, train acc=0.8855
test  loss=0.3954, test  acc=0.8936
Epoch 6:
train loss=0.3958, train acc=0.8916
test  loss=0.3688, test  acc=0.8979
Epoch 7:
train loss=0.3717, train acc=0.8969
test  loss=0.3489, test  acc=0.9002
Epoch 8:
train loss=0.3529, train acc=0.9008
test  loss=0.3331, test  acc=0.9045
Epoch 9:
train loss=0.3378, train acc=0.9038
test  loss=0.3201, test  acc=0.9074


### 7. PCA cu albire
Aplicați PCA cu albire pentru a obține 100 de trăsături.

In [36]:
pca_white = PCA(n_components=100, random_state=0, whiten=True)

X_train_pca_white = pca_white.fit_transform(X_train_norm).astype(np.float32)
X_test_pca_white = pca_white.transform(X_test_norm).astype(np.float32)

train_dataset_pca_white = PCA_Dataset(X_train_pca_white, y_train)
test_dataset_pca_white = PCA_Dataset(X_test_pca_white, y_test)

train_loader_pca_white = DataLoader(train_dataset_pca_white, batch_size=50, shuffle=True)
test_loader_pca_white = DataLoader(test_dataset_pca_white, batch_size=50, shuffle=False)

In [37]:
torch.manual_seed(0)
model = MLP_PCA().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader_pca_white, optimizer)
    test_loss, test_acc = test(model, test_loader_pca_white)
    print(f"Epoch {epoch}:")
    print(f"train loss={train_loss:.4f}, train acc={train_acc:.4f}")
    print(f"test  loss={test_loss:.4f}, test  acc={test_acc:.4f}")

Epoch 0:
train loss=2.2722, train acc=0.1600
test  loss=2.2124, test  acc=0.2438
Epoch 1:
train loss=2.1636, train acc=0.3436
test  loss=2.1040, test  acc=0.4571
Epoch 2:
train loss=2.0498, train acc=0.5331
test  loss=1.9821, test  acc=0.6123
Epoch 3:
train loss=1.9179, train acc=0.6489
test  loss=1.8371, test  acc=0.6989
Epoch 4:
train loss=1.7616, train acc=0.7160
test  loss=1.6663, test  acc=0.7554
Epoch 5:
train loss=1.5825, train acc=0.7600
test  loss=1.4764, test  acc=0.7903
Epoch 6:
train loss=1.3914, train acc=0.7913
test  loss=1.2819, test  acc=0.8131
Epoch 7:
train loss=1.2050, train acc=0.8143
test  loss=1.1009, test  acc=0.8350
Epoch 8:
train loss=1.0384, train acc=0.8326
test  loss=0.9451, test  acc=0.8527
Epoch 9:
train loss=0.8988, train acc=0.8463
test  loss=0.8181, test  acc=0.8618


### 8. Conexiune peste straturi (skip connection)
Antrenați o rețea cu cel puțin 2 straturi ascunse în care rezultatul primului strat ascuns contribuie la un strat mai avansat decât exact următorul.

In [38]:
class SkipMLP(nn.Module):
    def __init__(self, in_features=784, hidden=128, n_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, n_classes)

    def forward(self, x):
        h1 = F.relu(self.fc1(x))
        h2 = F.relu(self.fc2(h1))

        # h1 contribuie la un strat mai avansat decât următorul
        h3 = h2 + h1

        return self.fc3(h3)


torch.manual_seed(0)
model_skip = SkipMLP().to(device)
optimizer_skip = torch.optim.Adam(model_skip.parameters(), lr=1e-3)

for epoch in range(10):
    train_loss, train_acc = train(model_skip, train_loader, optimizer_skip)
    test_loss, test_acc = test(model_skip, test_loader)
    print(
        f"Epoch {epoch}: "
        f"train acc={train_acc:.4f}, test acc={test_acc:.4f}"
    )

Epoch 0: train acc=0.9179, test acc=0.9558
Epoch 1: train acc=0.9655, test acc=0.9699
Epoch 2: train acc=0.9758, test acc=0.9752
Epoch 3: train acc=0.9827, test acc=0.9762
Epoch 4: train acc=0.9861, test acc=0.9770
Epoch 5: train acc=0.9882, test acc=0.9785
Epoch 6: train acc=0.9905, test acc=0.9758
Epoch 7: train acc=0.9923, test acc=0.9782
Epoch 8: train acc=0.9928, test acc=0.9776
Epoch 9: train acc=0.9941, test acc=0.9775


### 9. Regularizare
Adăugați straturi de regularizare (`nn.Dropout(...)` sau `nn.BatchNorm1d(...)`) și antrenați rețeaua.

In [40]:
class RegularizedMLP(nn.Module):
    def __init__(self, in_features=784, hidden=128, n_classes=10):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(0)

model_reg = RegularizedMLP().to(device)
optimizer_reg = torch.optim.Adam(
    model_reg.parameters(),
    lr=1e-3
)

for epoch in range(10):
    train_loss, train_acc = train(
        model_reg,
        train_loader,
        optimizer_reg
    )
    test_loss, test_acc = test(
        model_reg,
        test_loader
    )

    print(
        f"Epoch {epoch}: "
        f"train acc={train_acc:.4f}, "
        f"test acc={test_acc:.4f}"
    )

Epoch 0: train acc=0.9110, test acc=0.9588
Epoch 1: train acc=0.9446, test acc=0.9668
Epoch 2: train acc=0.9543, test acc=0.9720
Epoch 3: train acc=0.9597, test acc=0.9710
Epoch 4: train acc=0.9621, test acc=0.9765
Epoch 5: train acc=0.9652, test acc=0.9762
Epoch 6: train acc=0.9683, test acc=0.9775
Epoch 7: train acc=0.9710, test acc=0.9780
Epoch 8: train acc=0.9719, test acc=0.9774
Epoch 9: train acc=0.9723, test acc=0.9787


### 10. Cel puțin 98% acuratețe cu cel mult 1M de parametri
Fără a folosi deloc datele de testare în procesul de antrenare, antrenați un model cu cel mult 1 milion de parametri antrenabili compus doar din straturi complet conectate care obține o acuratețe de cel puțin 98% pe datele de testare.

In [42]:
class MNIST98MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(0)
model = MNIST98MLP().to(device)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print("Parametri:", trainable_params)
assert trainable_params <= 1_000_000

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(15):
    train_loss, train_acc = train(model, train_loader, optimizer)
    test_loss, test_acc = test(model, test_loader)

    print(
        f"Epoch {epoch}: "
        f"train acc={train_acc:.4f}, test acc={test_acc:.4f}"
    )

Parametri: 243658
Epoch 0: train acc=0.8936, test acc=0.9623
Epoch 1: train acc=0.9404, test acc=0.9720
Epoch 2: train acc=0.9517, test acc=0.9774
Epoch 3: train acc=0.9572, test acc=0.9780
Epoch 4: train acc=0.9609, test acc=0.9771
Epoch 5: train acc=0.9650, test acc=0.9793
Epoch 6: train acc=0.9668, test acc=0.9824
Epoch 7: train acc=0.9681, test acc=0.9823
Epoch 8: train acc=0.9710, test acc=0.9826
Epoch 9: train acc=0.9734, test acc=0.9825
Epoch 10: train acc=0.9735, test acc=0.9811
Epoch 11: train acc=0.9759, test acc=0.9830
Epoch 12: train acc=0.9762, test acc=0.9815
Epoch 13: train acc=0.9759, test acc=0.9842
Epoch 14: train acc=0.9774, test acc=0.9837
